### 1.Agent循环
循环--模型与真实世界的第一道连接。  
语言模型能推理代码，但是碰不到真实世界。不能读写文件、跑测试、看报错。  
没有循环，每次工具调用你都得手动把结果粘贴回去，你自己就是那个循环。  

In [ ]:
import os
import subprocess

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"))


SYSTEM_PROMPT = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, do not explain."

TOOLS = [
    {
        "name": "bash",
        "description": "Run a shell command.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The shell command to run.",
                }
            }
        }
    }
]

def bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    
    
# -- 核心代码：一个调用工具的while循环
def agent_loop(messages: list):
    while True:
        responses = client.chat.completions.create(
            model="Pro/MiniMaxAI/MiniMax-M2.5",
            messages=messages,
            tools=TOOLS,
            max_tokens=8000,
            tool_choice="auto",
        )
        messages.append({"role": "assistant", "content": responses.choices[0].message.content})
        if not responses.choices[0].message.tool_calls:
            return
        
        print(responses)

if __name__ == "__main__":
    agent_loop([{"role": "user", "content": "ls"}])




TypeError: Completions.create() got an unexpected keyword argument 'system_prompt'